
**LOADING DATASET**


In [ ]:
# Check GPU is active
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("TensorFlow version:", tf.__version__)

# Install any missing libraries
!pip install -q opencv-python-headless matplotlib scikit-learn

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow version: 2.20.0


In [ ]:
import os
import json

# Step 1 - Kaggle config
kaggle_config = {
    "username": "YOUR_KAGGLE_USERNAME",
    "key": "YOUR_KAGGLE_KEY"
}
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_config, f)
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle configured!")

# Step 2 - Download
!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri
!kaggle datasets download -d mateuszbuda/lgg-mri-segmentation
print("Downloaded!")

# Step 3 - Unzip
!unzip -o -q brain-tumor-classification-mri.zip -d /content/classification
!unzip -o -q lgg-mri-segmentation.zip -d /content/segmentation
print(" Unzipped!")

# Step 4 - Verify
for folder in os.listdir('/content/classification/Training'):
    count = len(os.listdir(f'/content/classification/Training/{folder}'))
    print(f"  {folder}: {count} images")

Kaggle configured!
Dataset URL: https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
License(s): MIT
brain-tumor-classification-mri.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset URL: https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation
License(s): CC-BY-NC-SA-4.0
lgg-mri-segmentation.zip: Skipping, found more recently modified local copy (use --force to force download)
Downloaded!
--
 Unzipped!
  pituitary_tumor: 827 images
  meningioma_tumor: 822 images
  glioma_tumor: 826 images
  no_tumor: 395 images


In [ ]:
import os

# Check classification dataset
print("=== CLASSIFICATION DATASET ===")
for folder in os.listdir('/content/classification/Training'):
    count = len(os.listdir(f'/content/classification/Training/{folder}'))
    print(f"  {folder}: {count} images")

print("\nTesting:")
for folder in os.listdir('/content/classification/Testing'):
    count = len(os.listdir(f'/content/classification/Testing/{folder}'))
    print(f"  {folder}: {count} images")

# Check segmentation dataset
print("\n=== SEGMENTATION DATASET ===")
seg_path = '/content/segmentation/kaggle_3m'
folders = [f for f in os.listdir(seg_path) if os.path.isdir(os.path.join(seg_path, f))]
print(f"  Total patient folders: {len(folders)}")
# Count total images and masks
all_files = []
for f in folders:
    all_files += os.listdir(os.path.join(seg_path, f))
images = [f for f in all_files if '_mask' not in f and f.endswith('.tif')]
masks  = [f for f in all_files if '_mask' in f]
print(f"  Total MRI images: {len(images)}")
print(f"  Total masks: {len(masks)}")

=== CLASSIFICATION DATASET ===
  pituitary_tumor: 827 images
  meningioma_tumor: 822 images
  glioma_tumor: 826 images
  no_tumor: 395 images

Testing:
  pituitary_tumor: 74 images
  meningioma_tumor: 115 images
  glioma_tumor: 100 images
  no_tumor: 105 images

=== SEGMENTATION DATASET ===
  Total patient folders: 110
  Total MRI images: 3929
  Total masks: 3929


# **Training model**

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import seaborn as sns


In [ ]:
import os
from tensorflow.keras.utils import to_categorical

IMG_SIZE = 224
CLASS_NAMES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
TRAIN_PATH = '/content/classification/Training'
TEST_PATH  = '/content/classification/Testing'
BATCH_SIZE  = 32
SAVE_PATH   = '/content/drive/MyDrive/BrainTumorProject/best_classifier_v3.keras'

def load_images(base_path, class_names, img_size):
    images, labels = [], []
    for label, cls in enumerate(class_names):
        folder = os.path.join(base_path, cls)
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            img = cv2.imread(fpath)
            if img is None:
                continue
            img = cv2.resize(img, (img_size, img_size))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img / 255.0
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

print("Loading training images...")
X_train, y_train = load_images(TRAIN_PATH, CLASS_NAMES, IMG_SIZE)
print(f"Train: {X_train.shape}")

print("Loading testing images...")
X_test, y_test = load_images(TEST_PATH, CLASS_NAMES, IMG_SIZE)
print(f"Test: {X_test.shape}")

# Convert labels to categorical
y_train_cat = to_categorical(y_train, num_classes=len(CLASS_NAMES))
y_test_cat  = to_categorical(y_test,  num_classes=len(CLASS_NAMES))
print("Preprocessing done!")

Loading training images...


NameError: name 'cv2' is not defined

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ── Class weights ──────────────────────────────────────────
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))

print(f"X_train: {X_train.shape}")
print(f" X_test:  {X_test.shape}")
print(f" y_train_cat: {y_train_cat.shape}")
print(f" y_test_cat:  {y_test_cat.shape}")
print(f"⚖️  Class weights: {class_weights}")
print("\n All variables defined! Ready to train.")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, cls in enumerate(CLASS_NAMES):
    idx = np.where(y_train == i)[0][0]
    axes[i].imshow(X_train[idx])
    axes[i].set_title(cls)
    axes[i].axis('off')
plt.suptitle('Sample Training Images')
plt.tight_layout()
plt.show()

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
print(" Model built")

In [ ]:
for i, layer in enumerate(model.layers):
    print(f"[{i:3d}] {layer.name:40s} trainable={layer.trainable}")

In [ ]:
model.compile(
    optimizer=Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(" Model compiled!")
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

In [ ]:
#data_augumentation
datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)
print(" Augmentation ready!")

In [ ]:
print("Phase 1")

history1 = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=32),
    epochs=10,
    validation_data=(X_test, y_test_cat),
    verbose=1
)

In [ ]:
# Unfreeze top 8 layers of VGG16
for layer in base_model.layers[-8:]:
    layer.trainable = True

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(1e-5),  # very small lr for fine tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Phase 2:")
history2 = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=32),
    epochs=20,
    validation_data=(X_test, y_test_cat),
    verbose=1
)

this is overfitting

In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight

# CRITICAL FIX: EfficientNet needs [0,255] not [0,1]
# Your load_images divided by 255 — undo that here
X_train_eff = (X_train * 255).astype(np.float32)
X_test_eff  = (X_test  * 255).astype(np.float32)

y_train_cat = to_categorical(y_train, 4)
y_test_cat  = to_categorical(y_test,  4)

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))

print(f"X_train_eff: {X_train_eff.shape}, range [{X_train_eff.min():.0f}, {X_train_eff.max():.0f}]")
print(f"X_test_eff:  {X_test_eff.shape}")
print(f" Class weights: {class_weights}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

SAVE_PATH  = '/content/drive/MyDrive/BrainTumorProject/best_classifier_v3.keras'
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.10,
    horizontal_flip=True,
    width_shift_range=0.10,
    height_shift_range=0.10,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
    # NO rescale=1./255 here — EfficientNet handles it
)

tf.keras.backend.clear_session()

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
output = Dense(4, activation='softmax')(x)

model = Model(base_model.input, output)
model.compile(
    optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
print(f" Model ready")

def get_callbacks():
    return [
        ModelCheckpoint(SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1, mode='max'),
        EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=4, min_lr=1e-7, verbose=1)
    ]

# Phase 1
print("\n Phase 1...")
model.fit(
    datagen.flow(X_train_eff, y_train_cat, batch_size=BATCH_SIZE),
    epochs=25,
    validation_data=(X_test_eff, y_test_cat),
    callbacks=get_callbacks(),
    class_weight=class_weights,
    verbose=1
)

# Phase 2 — full fine-tune
print("\n Phase 2 — full fine-tune...")
for layer in base_model.layers:
    layer.trainable = True

model.compile(
    optimizer=AdamW(learning_rate=2e-5, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

model.fit(
    datagen.flow(X_train_eff, y_train_cat, batch_size=BATCH_SIZE),
    epochs=20,
    validation_data=(X_test_eff, y_test_cat),
    callbacks=get_callbacks(),
    class_weight=class_weights,
    verbose=1
)

In [ ]:
for i, layer in enumerate(model.layers):
    print(f"[{i}] {layer.name} — {type(layer).__name__}")

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import tensorflow as tf

SAVE_PATH = '/content/drive/MyDrive/BrainTumorProject/best_classifier_v3.keras'

model = load_model(SAVE_PATH)
print(" Best model loaded")

# EfficientNet layers are 0-237, custom head is 238-245
# Freeze first 187 layers, unfreeze last 50 EfficientNet layers + head
for layer in model.layers[:187]:
    layer.trainable = False
for layer in model.layers[187:]:
    layer.trainable = True

print(f"Trainable layers: {sum(1 for l in model.layers if l.trainable)}")

model.compile(
    optimizer=AdamW(learning_rate=5e-6, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

datagen_p2 = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.10,
    height_shift_range=0.10,
    brightness_range=[0.80, 1.20],
    fill_mode='nearest'
)

def get_callbacks_p2():
    return [
        ModelCheckpoint(SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1, mode='max'),
        EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, min_lr=1e-8, verbose=1)
    ]

print("\n Phase 2 (gentle, layers 187-245)...")
model.fit(
    datagen_p2.flow(X_train_eff, y_train_cat, batch_size=32),
    epochs=30,
    validation_data=(X_test_eff, y_test_cat),
    callbacks=get_callbacks_p2(),
    class_weight=class_weights,
    verbose=1
)

loss, acc = model.evaluate(X_test_eff, y_test_cat, verbose=0)
print(f"\n Final Test Accuracy: {acc*100:.2f}%")

In [ ]:
# ── STEP 1: Restore the TRUE best model (79.2%) ───────────
# Since Phase 2 kept overwriting, retrain Phase 1 from scratch
# but with one key fix — use the ORIGINAL saved best

# First check what accuracy the saved model actually gives
from tensorflow.keras.models import load_model
import numpy as np

model = load_model(SAVE_PATH)
loss, acc = model.evaluate(X_test_eff, y_test_cat, verbose=0)
print(f"Current saved model accuracy: {acc*100:.2f}%")


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

SAVE_PATH  = '/content/drive/MyDrive/BrainTumorProject/best_classifier_v3.keras'
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.10,
    horizontal_flip=True,
    width_shift_range=0.10,
    height_shift_range=0.10,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)

# ── Build fresh model ──────────────────────────────────────
tf.keras.backend.clear_session()

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze all except top 50
for layer in base_model.layers[:-50]:
    layer.trainable = False
for layer in base_model.layers[-50:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
output = Dense(4, activation='softmax')(x)

model = Model(base_model.input, output)
model.compile(
    optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
print(f" Model ready")

# ── Train — ONE phase, save to TWO separate paths ──────────
BEST_PATH  = '/content/drive/MyDrive/BrainTumorProject/best_classifier_v4.keras'
FINAL_PATH = '/content/drive/MyDrive/BrainTumorProject/final_classifier_v4.keras'

callbacks = [
    # Save to NEW file — never gets overwritten by a worse epoch
    ModelCheckpoint(BEST_PATH, monitor='val_accuracy',
                    save_best_only=True, verbose=1, mode='max'),
    EarlyStopping(monitor='val_accuracy', patience=12,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                      patience=4, min_lr=1e-7, verbose=1)
]

print("\n Training (top 50 layers unfrozen from start)...")
model.fit(
    datagen.flow(X_train_eff, y_train_cat, batch_size=BATCH_SIZE),
    epochs=40,           # more epochs, early stopping will handle it
    validation_data=(X_test_eff, y_test_cat),
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

# Save final separately — BEST_PATH stays untouched
model.save(FINAL_PATH)

loss, acc = model.evaluate(X_test_eff, y_test_cat, verbose=0)
print(f"\n Accuracy: {acc*100:.2f}%")
print(f"Best model safely saved to: {BEST_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import load_model

CLASS_NAMES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
BEST_PATH = '/content/drive/MyDrive/BrainTumorProject/best_classifier_v4.keras'

# Load best saved model
model = load_model(BEST_PATH)
loss, acc = model.evaluate(X_test_eff, y_test_cat, verbose=0)
print(f"Best Model Accuracy: {acc*100:.2f}%")

y_pred = np.argmax(model.predict(X_test_eff, verbose=0), axis=1)

print("\n Classification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Confusion Matrix — {acc*100:.1f}% Accuracy')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/BrainTumorProject/confusion_matrix_v4.png')
plt.show()

USED 83% accuracy model

In [ ]:
# ── STEP 1: CLAHE preprocessing (fixes glioma visibility) ──
import cv2
import numpy as np

def apply_clahe(images):
    result = []
    for img in images:
        lab = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        img_clahe = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB) / 255.0
        result.append(img_clahe)
    return np.array(result, dtype=np.float32)

print("Applying CLAHE...")
X_train_c = apply_clahe(X_train)   # use original X_train (0-1 range)
X_test_c  = apply_clahe(X_test)

# EfficientNet needs 0-255
X_train_eff2 = (X_train_c * 255).astype(np.float32)
X_test_eff2  = (X_test_c  * 255).astype(np.float32)
print(f" Done! Shape: {X_train_eff2.shape}")

In [ ]:
# ── STEP 2: Focal Loss definition ──────────────────────────
import tensorflow as tf

class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce     = -y_true * tf.math.log(y_pred)
        weight = tf.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))

In [ ]:
# ── STEP 3: Build + Train ───────────────────────────────────
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

BEST_PATH = '/content/drive/MyDrive/BrainTumorProject/best_classifier_v5.keras'

# Glioma boosted class weights
class_weights = {
    0: 5.0,   # glioma — main problem
    1: 1.0,   # meningioma
    2: 1.2,   # no_tumor
    3: 1.5    # pituitary
}

datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.10,
    height_shift_range=0.10,
    brightness_range=[0.80, 1.20],
    fill_mode='nearest'
)

tf.keras.backend.clear_session()

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers[:-50]:
    layer.trainable = False
for layer in base_model.layers[-50:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
output = Dense(4, activation='softmax')(x)

model = Model(base_model.input, output)
model.compile(
    optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
    loss=FocalLoss(gamma=2.0),
    metrics=['accuracy']
)

callbacks = [
    ModelCheckpoint(BEST_PATH, monitor='val_accuracy',
                    save_best_only=True, verbose=1, mode='max'),
    EarlyStopping(monitor='val_accuracy', patience=12,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                      patience=4, min_lr=1e-7, verbose=1)
]

print(" Training with Focal Loss + CLAHE + Glioma boost...")
model.fit(
    datagen.flow(X_train_eff2, y_train_cat, batch_size=32),
    epochs=40,
    validation_data=(X_test_eff2, y_test_cat),
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

loss, acc = model.evaluate(X_test_eff2, y_test_cat, verbose=0)
print(f"\n Accuracy: {acc*100:.2f}%")
print(f" Saved: {BEST_PATH}")

In [ ]:
# ── STEP 4: Evaluate ────────────────────────────────────────
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

y_pred = np.argmax(model.predict(X_test_eff2, verbose=0), axis=1)

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Confusion Matrix — {acc*100:.1f}%')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/BrainTumorProject/confusion_matrix_v5.png')
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
files = os.listdir('/content/drive/MyDrive/BrainTumorProject/')
print(files)

['best_classifier.h5', 'best_unet.keras', 'unet_predictions.png', 'best_classifier_v4.keras']


In [ ]:
!pip install huggingface_hub -q
from huggingface_hub import HfApi

HF_TOKEN = "YOUR_HF_TOKEN"
api = HfApi()

print("Uploading classifier...")
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/BrainTumorProject/best_classifier.h5",
    path_in_repo="best_classifier.h5",
    repo_id="Vanshika250/Brain-mri-analyzer",
    repo_type="space",
    token=HF_TOKEN
)
print(" Classifier uploaded!")

print("Uploading U-Net...")
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/BrainTumorProject/best_unet.keras",
    path_in_repo="best_unet.keras",
    repo_id="Vanshika250/Brain-mri-analyzer",
    repo_type="space",
    token=HF_TOKEN
)
print(" U-Net uploaded!")
print(" Done!")

Uploading classifier...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...roject/best_classifier.h5:  61%|######1   | 71.9MB /  117MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...orProject/best_unet.keras:  13%|#2        | 48.0MB /  377MB            

No files have been modified since last commit. Skipping to prevent empty commit.


 U-Net uploaded!
 Done!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, cv2, numpy as np, tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight

# ── Data Loading ───────────────────────────────────────────
IMG_SIZE    = 224
CLASS_NAMES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
TRAIN_PATH  = '/content/classification/Training'
TEST_PATH   = '/content/classification/Testing'

# Check if data exists
if not os.path.exists(TRAIN_PATH):
    import json
    kaggle_config = {"username": "YOUR_KAGGLE_USERNAME", "key": "YOUR_KAGGLE_KEY"}
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(kaggle_config, f)
    !chmod 600 /root/.kaggle/kaggle.json
    !kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri
    !unzip -o -q brain-tumor-classification-mri.zip -d /content/classification
    print(" Dataset downloaded!")

def load_images(base_path, class_names, img_size):
    images, labels = [], []
    for label, cls in enumerate(class_names):
        folder = os.path.join(base_path, cls)
        for fname in os.listdir(folder):
            img = cv2.imread(os.path.join(folder, fname))
            if img is None: continue
            img = cv2.cvtColor(cv2.resize(img, (img_size, img_size)),
                               cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(label)
    return np.array(images, dtype=np.float32), np.array(labels)

print(" Loading data...")
X_train, y_train = load_images(TRAIN_PATH, CLASS_NAMES, IMG_SIZE)
X_test,  y_test  = load_images(TEST_PATH,  CLASS_NAMES, IMG_SIZE)

# EfficientNet needs 0-255
X_train_eff = X_train.copy()
X_test_eff  = X_test.copy()

y_train_cat = to_categorical(y_train, 4)
y_test_cat  = to_categorical(y_test,  4)

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))
print(f" Train: {X_train_eff.shape} | Test: {X_test_eff.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Loading data...
 Train: (2870, 224, 224, 3) | Test: (394, 224, 224, 3)


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

SAVE_PATH  = '/content/drive/MyDrive/BrainTumorProject/best_classifier_v4.keras'
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rotation_range=15, zoom_range=0.10,
    horizontal_flip=True, width_shift_range=0.10,
    height_shift_range=0.10, brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)

tf.keras.backend.clear_session()
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers[:-50]:
    layer.trainable = False
for layer in base_model.layers[-50:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
output = Dense(4, activation='softmax')(x)

model = Model(base_model.input, output)
model.compile(
    optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
print(" Model ready!")

callbacks = [
    ModelCheckpoint(SAVE_PATH, monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                      patience=4, min_lr=1e-7, verbose=1)
]

print("\n🚀 Training...")
model.fit(
    datagen.flow(X_train_eff, y_train_cat, batch_size=BATCH_SIZE),
    epochs=40,
    validation_data=(X_test_eff, y_test_cat),
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

loss, acc = model.evaluate(X_test_eff, y_test_cat, verbose=0)
print(f"\nAccuracy: {acc*100:.2f}%")
print(f" Saved: {SAVE_PATH}")

 Model ready!

🚀 Training...
Epoch 1/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 737ms/step - accuracy: 0.4051 - loss: 1.6956
Epoch 1: val_accuracy improved from None to 0.46447, saving model to /content/drive/MyDrive/BrainTumorProject/best_classifier_v4.keras

Epoch 1: finished saving model to /content/drive/MyDrive/BrainTumorProject/best_classifier_v4.keras
90/90 ━━━━━━━━━━━━━━━━━━━━ 131s 966ms/step - accuracy: 0.4993 - loss: 1.4561 - val_accuracy: 0.4645 - val_loss: 1.2183 - learning_rate: 1.0000e-04
Epoch 2/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - accuracy: 0.6538 - loss: 1.1036
Epoch 2: val_accuracy improved from 0.46447 to 0.57614, saving model to /content/drive/MyDrive/BrainTumorProject/best_classifier_v4.keras

Epoch 2: finished saving model to /content/drive/MyDrive/BrainTumorProject/best_classifier_v4.keras
90/90 ━━━━━━━━━━━━━━━━━━━━ 40s 448ms/step - accuracy: 0.6767 - loss: 1.0542 - val_accuracy: 0.5761 - val_loss: 1.1213 - learning_rate: 1.0000e-04
Epoch 3/40
90/90 ━━━━━━━━━━━━━━

In [ ]:
from huggingface_hub import HfApi

HF_TOKEN = "YOUR_HF_TOKEN"
api = HfApi()

print("Uploading classifier v4...")
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/BrainTumorProject/best_classifier_v4.keras",
    path_in_repo="best_classifier_v4.keras",
    repo_id="Vanshika250/Brain-mri-analyzer",
    repo_type="space",
    token=HF_TOKEN
)
print(" Done!")

Uploading classifier v4...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../best_classifier_v4.keras:   1%|1         |  580kB / 41.7MB            

 Done!
